# Ingeniero de boxes para simracing

**Carlos Peñalver · IA Generativa**

Gemini, Chroma y LangGraph: detección de simulador, RAG y memoria. Las respuestas se conservan como salidas reales de cinco celdas ejecutadas.


## 0. Configuración

Usa el kernel `Python 3.12 (Proyecto Simracing)` y configura `.env` en la raíz del proyecto. Leer las salidas guardadas no consume API; volver a ejecutar los ejemplos sí.


In [1]:
from pathlib import Path
from uuid import uuid4
import json
import os
import sys

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from retrieval import GeminiRetrievalEmbeddings, index_documents
from agent import crear_grafo

load_dotenv(ROOT / ".env", override=True)
CORPUS_DIR = ROOT / "data/corpus"
CHROMA_DIR = ROOT / "data/chroma"
SYSTEM_PROMPT = (ROOT / "prompts/system.md").read_text(encoding="utf-8")


## 1. Corpus

75 fichas de LMU (Ultimate Setup Hub), iRacing (Chris Haye) y ACC (Driver61). Cada ficha conserva el texto y sus metadatos; fuentes enlazadas en el [README](../README.md).


In [2]:
records = [json.loads(line) for line in
           (CORPUS_DIR / "mvp/preparado/corpus.jsonl").read_text(encoding="utf-8").splitlines()]
documents = [
    Document(page_content=record["text"], metadata={
        key: value for key, value in record.items() if key != "text"
    })
    for record in records
]


## 2. Embeddings y Chroma

Se reutiliza el índice de `data/chroma/`, sin recalcular embeddings del corpus. Cada búsqueda del agente sí genera un embedding de consulta.


In [3]:
MODEL_EMBEDDING = os.environ["GEMINI_EMBEDDING_MODEL"]
llm_embedding = GeminiRetrievalEmbeddings(
    model=MODEL_EMBEDDING,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
    request_options={"timeout": 30000, "retry_options": {"attempts": 1}},
)


In [4]:
vectorstore = index_documents(documents, llm_embedding, CHROMA_DIR)


## 3. System prompt

`prompts/system.md` define la voz del ingeniero, el respaldo documental, los límites y una prueba de ajuste cada vez. El clasificador usa `prompts/classify.md`; la orientación sin corpus para otros simuladores tiene su propio prompt. Justificación en el README.


## 4. Agente en LangGraph

`START → clasificar → [lmu, iracing, acc, distinto] → END`

Gemini clasifica la conversación y cada rama especializada recupera documentos de su simulador. `distinto` pregunta si falta contexto, reconduce temas ajenos u ofrece orientación general con aviso para otros juegos. El código compartido está en `agent.py`.

Cada consulta especializada utiliza clasificación, búsqueda y generación. La temperatura 1.0 sigue la recomendación del proveedor para Gemini 3; no representa un porcentaje de uso del corpus.


In [5]:
MODEL = os.environ["GEMINI_MODEL"]
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
    temperature=1.0,
    max_output_tokens=1800,
    max_retries=3,
    timeout=45,
)
checkpointer = InMemorySaver()
app = crear_grafo(vectorstore, llm, SYSTEM_PROMPT, checkpointer=checkpointer)


## 5. Memoria por conversación

El mismo `thread_id` conserva el contexto y el historial con roles; otro inicia una sesión independiente. Los cambios de coche o simulador se detectan al conversar y se confirman antes de retirar los reglajes anteriores. La memoria se pierde al reiniciar el kernel.

Los ejemplos 1 y 2 de abajo comparten conversación para demostrar un seguimiento sin cambios aplicados. Los demás usan sesiones independientes.


## 6. Chat interactivo

Habla con el agente indicando simulador, coche y síntoma. Si falta el simulador, lo preguntará. Confirma o cancela los cambios conversando; solo existen `/nuevo` y `/salir`.

Al ejecutar todo el notebook, escribe `/salir` para continuar con los cinco ejemplos. Antes de compartirlo, limpia cualquier conversación personal de esta celda.


In [6]:
def iniciar_chat(app, ver_fuentes=True, leer=None, escribir=None):
    """Chat con detección automática; no selecciona un simulador por defecto."""
    leer = input if leer is None else leer
    escribir = (lambda texto: display(Markdown(texto))) if escribir is None else escribir
    config_chat = {"configurable": {"thread_id": uuid4().hex}}
    escribir("**Chat de boxes**. Cuéntame en qué simulador conduces y qué notas. "
             "Puedes confirmar o cancelar los cambios conversando. Comandos: `/nuevo` y `/salir`.")
    while True:
        try:
            texto = leer("Piloto > ").strip()
        except (EOFError, KeyboardInterrupt):
            escribir("Chat cerrado.")
            break
        if not texto:
            continue
        if texto.lower() == "/salir":
            escribir("Chat cerrado.")
            break
        if texto.lower() == "/nuevo":
            config_chat = {"configurable": {"thread_id": uuid4().hex}}
            escribir("Conversación nueva, sin simulador ni coche seleccionados.")
            continue
        if texto.startswith("/"):
            escribir("Usa `/nuevo` o `/salir`. Indica el simulador, el coche y los cambios con tus propias palabras.")
            continue
        try:
            resultado = app.invoke({"pregunta": texto}, config=config_chat)
        except KeyboardInterrupt:
            escribir("Chat interrumpido.")
            break
        except Exception:
            escribir("No se ha podido completar la respuesta. Revisa la conexión o la cuota de Gemini y prueba más tarde.")
            continue
        escribir(resultado["respuesta"] if ver_fuentes else resultado["respuesta_piloto"])
        if ver_fuentes and resultado["fuentes"]:
            escribir("**Fuentes para revisión**\n\n" + "\n".join(
                f"- [{f['id']}]({f['url']}) — {f['locator']}" for f in resultado["fuentes"]
            ))
    return config_chat


def mostrar_respuesta(resultado):
    """Muestra la salida real del agente y conserva sus referencias para revisión."""
    display(Markdown(resultado["respuesta"]))
    if resultado["fuentes"]:
        display(Markdown("**Fuentes:** " + ", ".join(
            f"[{f['id']}]({f['url']})" for f in resultado["fuentes"]
        )))


In [7]:
config_chat = iniciar_chat(app, ver_fuentes=True)


**Chat de boxes**. Cuéntame en qué simulador conduces y qué notas. Puedes confirmar o cancelar los cambios conversando. Comandos: `/nuevo` y `/salir`.

Chat cerrado.

### Cinco ejemplos documentados

Consultas ficticias del piloto ejecutadas realmente con Gemini. Las respuestas siguientes son salidas de las celdas, no transcripciones en Markdown. Los dos primeros ejemplos comparten sesión; los demás empiezan otra. El último comprueba el límite de la documentación.

Las citas permiten revisar el respaldo, pero no garantizan que cada explicación sea correcta ni validan ajustes en pista. Persisten explicaciones con respaldo parcial; estos ejemplos no son una validación física.


In [8]:
# 1. LMU: primera propuesta
config_demo = {"configurable": {"thread_id": uuid4().hex}}
resultado = app.invoke(
    {"pregunta": (
        "En LMU, mi Ferrari GT3 sobrevira al acelerar a la salida de curvas lentas. ¿Qué "
        "primera prueba harías? Responde brevemente."
    )},
    config=config_demo,
)
mostrar_respuesta(resultado)


Si asumiendo que tu trazada es correcta el coche sobrevira al acelerar en curvas lentas, el problema es que el peso no se está transfiriendo al eje trasero con la suficiente rapidez o eficacia [LMU_USH_24].

Empezaría por **aumentar la altura delantera (front ride height)**. Al realizar este ajuste, reducimos el "rake" (inclinación) del coche, lo que ayuda a que el balance de pesos se desplace hacia atrás con mayor facilidad al acelerar.

Si tras probar este cambio en incrementos pequeños no notas una mejora en la tracción a la salida, revierte el ajuste y prueba la siguiente opción de la lista: **reducir la altura trasera (rear ride height)** [LMU_USH_24].

¿Has confirmado ya tu trazada o notas que el sobreviraje aparece incluso siendo suave con el acelerador?

**Fuentes:** [LMU_USH_24](https://ultimatesetuphub.com/race-engineer/n5n2n0)

In [9]:
# 2. Seguimiento: todavía no se ha aplicado la propuesta
resultado = app.invoke(
    {"pregunta": (
        "Todavía no he cambiado nada. ¿Qué debería observar al probar solo tu primera "
        "propuesta? Responde brevemente."
    )},
    config=config_demo,
)
mostrar_respuesta(resultado)


Al aumentar la altura delantera, el objetivo es reducir la inclinación (rake) del coche para facilitar que el peso se desplace hacia atrás con mayor rapidez al abrir gas [LMU_USH_24].

Lo que debes observar es una **mayor estabilidad y agarre en el eje trasero** justo en el momento en que aplicas potencia a la salida de las curvas lentas. El coche debería sentirse menos nervioso y mostrar menos tendencia a "deslizarse" de atrás cuando entregas par motor.

Si tras aplicar incrementos pequeños notas que el coche responde mejor y la rotación excesiva disminuye, conserva el ajuste. Si, por el contrario, sientes que el morro pierde precisión en la entrada o el equilibrio general no mejora, revierte el cambio y pasaremos a la siguiente prueba: reducir la altura trasera.

**Fuentes:** [LMU_USH_24](https://ultimatesetuphub.com/race-engineer/n5n2n0)

In [10]:
# 3. iRacing: sesión independiente
config_iracing = {"configurable": {"thread_id": uuid4().hex}}
resultado = app.invoke(
    {"pregunta": (
        "En iRacing bloqueo a menudo los neumáticos delanteros al frenar. ¿Qué cambio de "
        "reparto de frenada probarías? Responde brevemente."
    )},
    config=config_iracing,
)
mostrar_respuesta(resultado)


Empezaría por desplazar el reparto de frenada (brake bias) hacia atrás [IR_CHRIS_HAYE_02]. 

Esto apunta a un exceso de potencia de frenado en el eje delantero que supera la capacidad de agarre de los neumáticos. Fíjate en cómo reacciona el coche en la entrada de las curvas tras el ajuste: el objetivo es que las ruedas delanteras se bloqueen justo antes que las traseras para mantener la estabilidad óptima [IR_CHRIS_HAYE_20]. Si al realizar este cambio notas que el coche pierde el control al frenar, revierte el ajuste y revisa si el problema persiste o si necesitas otra aproximación [IR_CHRIS_HAYE_01].

¿Qué coche estás pilotando exactamente para poder ajustar mejor las recomendaciones a sus capacidades?

**Fuentes:** [IR_CHRIS_HAYE_02](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/), [IR_CHRIS_HAYE_20](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/), [IR_CHRIS_HAYE_01](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/)

In [11]:
# 4. ACC: sesión independiente
config_acc = {"configurable": {"thread_id": uuid4().hex}}
resultado = app.invoke(
    {"pregunta": (
        "En Assetto Corsa Competizione noto el coche incómodo al pasar por los pianos. "
        "¿Qué primer ajuste probarías? Responde brevemente."
    )},
    config=config_acc,
)
mostrar_respuesta(resultado)


Empezaría por reducir los valores de **Fast Bump** y **Fast Rebound** en tus amortiguadores. Si el coche se siente rígido o nervioso al atacar los pianos, esto apunta a que la suspensión no está absorbiendo el impacto con la rapidez necesaria [ACC_DRIVER61_18].

Al realizar este ajuste, busca que el coche se asiente mejor al contacto con el piano y pierda esa sensación de rebote seco. Si tras probarlo el comportamiento no mejora o notas que el coche pierde control en zonas bacheadas, revierte el cambio a los valores originales. Ten en cuenta que esta es una recomendación basada en documentación histórica para ACC y no puedo garantizar que sea óptima para tu modelo de coche específico o la versión actual del juego.

¿Qué valores tienes configurados actualmente en Fast Bump y Fast Rebound?

**Fuentes:** [ACC_DRIVER61_18](https://driver61.com/wp-content/uploads/2021/04/Driver61-Assetto-Corsa-Competizione-Setup-Guide-V1.pdf)

In [12]:
# 5. Falta de evidencia: no inventar una presión exacta
config_limite = {"configurable": {"thread_id": uuid4().hex}}
resultado = app.invoke(
    {"pregunta": (
        "¿Qué presión exacta es óptima para mi Ferrari GT3 en la versión actual de LMU? "
        "Responde brevemente."
    )},
    config=config_limite,
)
mostrar_respuesta(resultado)


No existe una cifra exacta de presión óptima para el Ferrari GT3 en Le Mans Ultimate. Los datos disponibles no proporcionan valores específicos para esta versión, ya que las recomendaciones se centran en el ajuste relativo según la temperatura del neumático [LMU_USH_10, LMU_USH_11].

Para gestionar la temperatura de los neumáticos en pista:

*   **Si el neumático está demasiado caliente:** Aumentar las presiones reduce la superficie de contacto y, por tanto, la acumulación de calor [LMU_USH_10].
*   **Si el neumático está demasiado frío:** Disminuir las presiones permite que el neumático flexione más bajo carga, lo que ayuda a elevar su temperatura [LMU_USH_11].

Realiza cambios incrementales pequeños y prueba el efecto antes de volver a ajustar. ¿Qué temperaturas estás registrando actualmente en tus neumáticos?

**Fuentes:** [LMU_USH_11](https://ultimatesetuphub.com/race-engineer/n3n4), [LMU_USH_10](https://ultimatesetuphub.com/race-engineer/n3n3)

## 7. Próximo paso

Recoger la siguiente propuesta de mejora y después preparar la web en Streamlit sobre el mismo núcleo.
